# Microservices Simulation – Search Engine
This notebook implements:
- Logical operators (AND / OR) for querying
- Simple ranking of documents based on matched terms

In [1]:
class IndexService:
    def __init__(self):
        self.documents = {}
        self.index = {}

    def add_document(self, doc_data):
        doc_id = str(len(self.documents) + 1)
        self.documents[doc_id] = {**doc_data, 'id': doc_id}
        words = doc_data['content'].lower().split()
        for word in words:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)
        return self.documents[doc_id]

    def get_document(self, doc_id):
        return self.documents.get(doc_id)

    def search_word(self, word):
        return list(self.index.get(word.lower(), set()))


In [2]:
class QueryService:
    def __init__(self, index_service):
        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):
        try:
            query_id = str(len(self.queries) + 1)
            search_terms = query_data['terms']
            logic = query_data.get('logic', 'AND').upper()

            results = set()
            for i, term in enumerate(search_terms):
                doc_ids = set(self.index_service.search_word(term))
                if i == 0:
                    results = doc_ids
                else:
                    if logic == 'AND':
                        results &= doc_ids
                    elif logic == 'OR':
                        results |= doc_ids

            query = {
                'id': query_id,
                'terms': search_terms,
                'logic': logic,
                'results': list(results),
                'timestamp': query_data.get('timestamp', 'now')
            }
            self.queries[query_id] = query
            return query
        except Exception as e:
            return {'error': str(e)}


In [3]:
class ResultService:
    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service
        self.results = {}

    def format_results(self, query_id):
        try:
            query = self.query_service.queries.get(query_id)
            if not query:
                return {'error': 'Query not found'}

            search_terms = query['terms']
            formatted_results = []
            for doc_id in query['results']:
                doc = self.index_service.get_document(doc_id)
                if doc:
                    content = doc['content'].lower()
                    rank = sum(1 for term in search_terms if term.lower() in content)
                    formatted_results.append({
                        'doc_id': doc_id,
                        'title': doc['title'],
                        'snippet': doc['content'][:100] + '...',
                        'rank': rank
                    })

            # Sort by rank descending
            formatted_results.sort(key=lambda x: x['rank'], reverse=True)

            result_id = str(len(self.results) + 1)
            result = {
                'id': result_id,
                'query_id': query_id,
                'formatted_results': formatted_results,
                'count': len(formatted_results)
            }
            self.results[result_id] = result
            return result
        except Exception as e:
            return {'error': str(e)}


In [4]:
def main():
    index_service = IndexService()
    query_service = QueryService(index_service)
    result_service = ResultService(index_service, query_service)

    # Add documents
    index_service.add_document({
        'title': 'Python Programming',
        'content': 'Python is a popular programming language for cloud computing'
    })
    index_service.add_document({
        'title': 'Cloud Services',
        'content': 'Cloud computing enables scalable microservices architecture'
    })
    index_service.add_document({
        'title': 'Serverless Functions',
        'content': 'Functions as a Service allow cloud-based deployments'
    })

    # Run AND query
    and_query = query_service.create_query({'terms': ['cloud', 'computing'], 'logic': 'AND'})
    print("AND Query Results:", and_query)
    print("Ranked Results:", result_service.format_results(and_query['id']))

    # Run OR query
    or_query = query_service.create_query({'terms': ['python', 'serverless'], 'logic': 'OR'})
    print("OR Query Results:", or_query)
    print("Ranked Results:", result_service.format_results(or_query['id']))

main()

AND Query Results: {'id': '1', 'terms': ['cloud', 'computing'], 'logic': 'AND', 'results': ['1', '2'], 'timestamp': 'now'}
Ranked Results: {'id': '1', 'query_id': '1', 'formatted_results': [{'doc_id': '1', 'title': 'Python Programming', 'snippet': 'Python is a popular programming language for cloud computing...', 'rank': 2}, {'doc_id': '2', 'title': 'Cloud Services', 'snippet': 'Cloud computing enables scalable microservices architecture...', 'rank': 2}], 'count': 2}
OR Query Results: {'id': '2', 'terms': ['python', 'serverless'], 'logic': 'OR', 'results': ['1'], 'timestamp': 'now'}
Ranked Results: {'id': '2', 'query_id': '2', 'formatted_results': [{'doc_id': '1', 'title': 'Python Programming', 'snippet': 'Python is a popular programming language for cloud computing...', 'rank': 1}], 'count': 1}
